# AAAI-27 rebuttal — paper_full / restored pilot

原始 `Python/`、Gaussian R ESS、MFVB 实现不改。可选 paper_full 或恢复的快速 pilot；pilot 使用原版 .5 到 .1 温度日程，不能把两者差异完全归因于迭代数。


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os, sys, shutil, importlib
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "rebuttal27").is_dir():
    ROOT = ROOT / "AAAI27_rebuttal_verified"
if not (ROOT / "rebuttal27").is_dir():
    raise FileNotFoundError("Notebook must run from AAAI27_rebuttal_verified or its parent")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import rebuttal27.runner as runner
assert Path(runner.__file__).resolve() == (ROOT / "rebuttal27" / "runner.py").resolve(), "Restart Kernel: old runner loaded"
importlib.reload(runner)
runner.verify_untouched_sources()
print("Verified original source SHA256; project:", ROOT)


## Optimization profile

`pilot` 是旧版快速训练配置；`paper_full` 是 Table 2 完整配置。 The default flow has `K_q=64, K_g=4` (68 semantic cycles).

`paper_full`: fixed `tau=0.8`. The previously used `pilot`: `tau_start=0.5, tau_end=0.1`, a different temperature schedule. Do not interpret this as a training-duration-only comparison.


In [ ]:
TASK = "depth"  # depth / mfvb / difficult / correlation / poisson / boston / gse
SEED = 400
PROFILE = "pilot"
USE_MCMC = True
RSCRIPT = r"C:\Program Files\R\R-4.5.1\bin\Rscript.exe"
MCMC_N = 10000  # 原本 MCMC 采样预算保留；独立于 Table 2 的 VI 配置
MCMC_BURN = 2000
MCMC_SMAX = 100
BETA_EPS = 0.05  # 与用户已完成的 ESS 运行一致
THREADS = 8
MFVB_EPOCHS = 600
MFVB_SLAB_VAR = 0.5  # EM 的初始值，非固定方差
MFVB_SLAB_MODE = "em"
MFVB_RESTARTS = 3
COMMON = dict(seed=SEED, profile=PROFILE, mcmc=USE_MCMC, rscript=RSCRIPT,
              mcmc_n=MCMC_N, mcmc_burn=MCMC_BURN, mcmc_smax=MCMC_SMAX,
              beta_eps=BETA_EPS, threads=THREADS, mfvb_epochs=MFVB_EPOCHS,
              mfvb_slab_var=MFVB_SLAB_VAR, mfvb_slab_mode=MFVB_SLAB_MODE,
              mfvb_restarts=MFVB_RESTARTS)
if USE_MCMC and not (shutil.which(RSCRIPT) or Path(RSCRIPT).is_file()):
    raise FileNotFoundError("Rscript not found. Set RSCRIPT to the installed Rscript.exe")
print("Task:", TASK, "| profile:", PROFILE, "| default paper depth: 64+4")


## 1. Flow depth ablation（所有 depth 使用所选 PROFILE 的训练日程）


In [ ]:
if TASK == 'depth':
    result = runner.run_depth(depths=(4,8,16,32,64,68), setting='baseline', **COMMON)
    display(result)

## 2. Traditional point-mass spike-and-slab MFVB


In [ ]:
if TASK == 'mfvb':
    result = runner.run_mfvb(setting='baseline', family='gaussian', depth=68, **COMMON)
    display(result)

## 3. Difficult Gaussian: MFVB / RAT / Odd-even / Thinner


In [ ]:
if TASK == 'difficult':
    result = runner.run_difficult(settings=('weak_signal','low_snr','p_gt_n'), depth=68, **COMMON)
    display(result)

## 4. Correlated predictors（AR(1)）


In [ ]:
if TASK == 'correlation':
    result = runner.run_correlation(rhos=(.25,.50,.75), depth=68, **COMMON)
    display(result)

## 5. Poisson count regression


In [ ]:
if TASK == 'poisson':
    result = runner.run_poisson(settings=('baseline','weak_signal','low_snr','n_gt_p','p_gt_n'), depth=68, **COMMON)
    display(result)

## 6. Boston（Gaussian）


In [ ]:
if TASK == 'boston':
    result = runner.run_boston(n_noise=12, **COMMON)
    display(result)

## 7. GSE32448（microarray；tumor vs benign/normal logistic）


In [ ]:
if TASK == 'gse':
    result = runner.run_gse(n_genes=40, n_noise=30, outcome='tumor', **COMMON)
    display(result)

## 结果说明

一个 task 输出一个 CSV。SKL 不可估计的分量跳过，报告成功估计项的中位数，不再输出 `valid/total` 列；全部不可估计时保持 NaN。
原始 R ESS 的缓存按 likelihood family 隔离。跨先验的 posterior metrics 使用 `cross_prior_*` 前缀。


In [ ]:
if 'result' in globals():
    show = ['dataset','method','depth','n_coupling_transforms','wall_seconds',
            'support_size','TPR','FPR','test_nll','test_mse',
            'noise_FPR','cross_prior_effective_PIP_RMSE']
    display(result[[k for k in show if k in result.columns]])